In [5]:
# Cell 1 — Define target + control field centers, and verify Galactic latitude (b)

import numpy as np
import pandas as pd

# If you have astropy (recommended):
from astropy.coordinates import SkyCoord
import astropy.units as u

# -----------------------------
# CONFIG
# -----------------------------
HALF_SIZE_DEG = 0.5  # 1°×1° box, centered

# Target field center (J2000)
TARGET = ("target", 32.405833, -4.642111)

# Controls chosen far away but intended to match Galactic latitude (precomputed)
CONTROL_CENTERS = [
    ("glon_080", 353.381110, -4.590060),
    ("glon_085", 355.375477, -3.178909),
    ("glon_090", 357.466454, -1.919136),
    ("glon_095", 359.644412, -0.819254),
    ("glon_100",   1.898647,  0.113008),
    ("glon_105",   4.217379,  0.870862),
    ("glon_110",   6.587807,  1.448624),
    ("glon_115",   8.996243,  1.841851),
    ("glon_120",  11.428305,  2.047465),
    ("glon_125",  13.869163,  2.063840),
    ("glon_130",  16.303813,  1.890845),
    ("glon_135",  18.717374,  1.529851),
    ("glon_140",  21.095369,  0.983691),
]

# -----------------------------
# Build a table of fields
# -----------------------------
fields = [TARGET] + CONTROL_CENTERS
df_fields = pd.DataFrame(fields, columns=["field_id", "ra_deg", "dec_deg"])

# Convert to Galactic l,b
coords = SkyCoord(ra=df_fields["ra_deg"].values * u.deg,
                  dec=df_fields["dec_deg"].values * u.deg,
                  frame="icrs")

df_fields["l_deg"] = coords.galactic.l.deg
df_fields["b_deg"] = coords.galactic.b.deg

# Compare b to target b
b0 = df_fields.loc[df_fields["field_id"] == "target", "b_deg"].iloc[0]
df_fields["delta_b_deg"] = df_fields["b_deg"] - b0
df_fields["abs_delta_b_deg"] = np.abs(df_fields["delta_b_deg"])

# Sort by abs delta_b to see worst offenders first
df_fields_sorted = df_fields.sort_values("abs_delta_b_deg", ascending=False).reset_index(drop=True)

display(df_fields_sorted)

print(f"Target Galactic latitude b = {b0:.6f} deg")
print(f"Max |Δb| among controls = {df_fields[df_fields['field_id']!='target']['abs_delta_b_deg'].max():.6f} deg")
print(f"Median |Δb| among controls = {df_fields[df_fields['field_id']!='target']['abs_delta_b_deg'].median():.6f} deg")

,field_id,ra_deg,dec_deg,l_deg,b_deg,delta_b_deg,abs_delta_b_deg
0,glon_080,353.381110,-4.590060,80.000011,-60.791691,-2.644260e-06,2.644260e-06
1,glon_095,359.644412,-0.819254,95.000012,-60.791690,-2.174675e-06,2.174675e-06
2,glon_085,355.375477,-3.178909,85.000011,-60.791690,-1.899046e-06,1.899046e-06
3,glon_090,357.466454,-1.919136,90.000012,-60.791690,-1.828737e-06,1.828737e-06
4,glon_105,4.217379,0.870862,105.000010,-60.791689,-1.359694e-06,1.359694e-06
5,glon_100,1.898647,0.113008,100.000010,-60.791689,-1.237323e-06,1.237323e-06
6,glon_120,11.428305,2.047465,120.000009,-60.791689,-1.081309e-06,1.081309e-06
7,glon_125,13.869163,2.063840,125.000010,-60.791689,-8.724504e-07,8.724504e-07
8,glon_130,16.303813,1.890845,130.000010,-60.791689,-6.632705e-07,6.632705e-07
9,glon_110,6.587807,1.448624,110.000011,-60.791689,-6.444923e-07,6.444923e-07


Target Galactic latitude b = -60.791688 deg
Max |Δb| among controls = 0.000003 deg
Median |Δb| among controls = 0.000001 deg


In [6]:
# Cell 2 — Download SDSS CSVs for each field box (1°×1°), using SkyServer SQL

import time
from dataclasses import dataclass
from pathlib import Path
from urllib.parse import urlencode

import requests

# -----------------------------
# CONFIG
# -----------------------------
DATA_RELEASE = "dr17"
BASE_URL = f"https://skyserver.sdss.org/{DATA_RELEASE}/SkyServerWS/SearchTools/SqlSearch"

OUTDIR = Path("fields")
OUTDIR.mkdir(parents=True, exist_ok=True)

SLEEP_SECONDS = 1.0

# Optional: add a light SQL cut to keep files smaller (uncomment if needed)
# EXTRA_WHERE = "AND p.psfMag_r BETWEEN 14 AND 22.2"
EXTRA_WHERE = ""

# -----------------------------
# Field boxes
# -----------------------------
@dataclass(frozen=True)
class FieldBox:
    field_id: str
    ra_min: float
    ra_max: float
    dec_min: float
    dec_max: float

def wrap_ra(ra: float) -> float:
    return ra % 360.0

def make_box(field_id: str, ra_center: float, dec_center: float, half_size: float) -> FieldBox:
    ra_center = wrap_ra(ra_center)
    ra_min = ra_center - half_size
    ra_max = ra_center + half_size
    dec_min = dec_center - half_size
    dec_max = dec_center + half_size
    return FieldBox(field_id=field_id, ra_min=ra_min, ra_max=ra_max, dec_min=dec_min, dec_max=dec_max)

def build_sql(box: FieldBox) -> str:
    # RA condition with wrap handling
    if box.ra_min < 0:
        ra_cond = f"(p.ra BETWEEN 0 AND {box.ra_max:.6f} OR p.ra BETWEEN {360.0 + box.ra_min:.6f} AND 360)"
    elif box.ra_max >= 360:
        ra_cond = f"(p.ra BETWEEN {box.ra_min:.6f} AND 360 OR p.ra BETWEEN 0 AND {box.ra_max - 360.0:.6f})"
    else:
        ra_cond = f"(p.ra BETWEEN {box.ra_min:.6f} AND {box.ra_max:.6f})"

    sql = f"""
SELECT
    p.objid,
    p.ra, p.dec,
    p.psfMag_u, p.psfMag_g, p.psfMag_r, p.psfMag_i, p.psfMag_z,
    p.psfMagErr_u, p.psfMagErr_g, p.psfMagErr_r, p.psfMagErr_i, p.psfMagErr_z,
    p.extinction_u, p.extinction_g, p.extinction_r, p.extinction_i, p.extinction_z,
    p.type,
    p.mode
FROM PhotoPrimary AS p
WHERE
    {ra_cond}
AND p.dec BETWEEN {box.dec_min:.6f} AND {box.dec_max:.6f}
AND p.type = 6
AND p.mode = 1
AND p.clean = 1
{EXTRA_WHERE}
"""
    return " ".join(sql.split())

def fetch_csv(sql: str, outfile: Path, timeout: int = 300) -> None:
    params = {"cmd": sql, "format": "csv"}
    url = f"{BASE_URL}?{urlencode(params)}"
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    outfile.write_bytes(r.content)

# -----------------------------
# Build list of fields from df_fields (created in Cell 1)
# -----------------------------
# Expect df_fields to exist from Cell 1
boxes = []
for _, row in df_fields.iterrows():
    boxes.append(make_box(row["field_id"], float(row["ra_deg"]), float(row["dec_deg"]), HALF_SIZE_DEG))

print(f"Downloading {len(boxes)} fields ({len(boxes)-1} controls) from {BASE_URL}")
print(f"Output directory: {OUTDIR.resolve()}\n")

for i, box in enumerate(boxes, start=1):
    sql = build_sql(box)
    outfile = OUTDIR / f"sdss_{box.field_id}.csv"
    print(f"[{i:02d}/{len(boxes):02d}] {box.field_id}: "
          f"RA[{box.ra_min:.3f},{box.ra_max:.3f}] Dec[{box.dec_min:.3f},{box.dec_max:.3f}] -> {outfile.name}")

    try:
        fetch_csv(sql, outfile)
    except requests.HTTPError as e:
        debug_sql = OUTDIR / f"FAILED_{box.field_id}.sql.txt"
        debug_sql.write_text(sql)
        print(f"  ERROR: {e}\n  Saved failing SQL to: {debug_sql.name}")
    except Exception as e:
        debug_sql = OUTDIR / f"FAILED_{box.field_id}.sql.txt"
        debug_sql.write_text(sql)
        print(f"  ERROR: {e}\n  Saved failing SQL to: {debug_sql.name}")

    time.sleep(SLEEP_SECONDS)

print("\nDone.")

Output directory: /Users/winstonzhang/Code/GitHub/ASTR-5140/project2/sdss_data/fields

[01/14] target: RA[31.906,32.906] Dec[-5.142,-4.142] -> sdss_target.csv
[02/14] glon_080: RA[352.881,353.881] Dec[-5.090,-4.090] -> sdss_glon_080.csv
[03/14] glon_085: RA[354.875,355.875] Dec[-3.679,-2.679] -> sdss_glon_085.csv
[04/14] glon_090: RA[356.966,357.966] Dec[-2.419,-1.419] -> sdss_glon_090.csv
[05/14] glon_095: RA[359.144,360.144] Dec[-1.319,-0.319] -> sdss_glon_095.csv
[06/14] glon_100: RA[1.399,2.399] Dec[-0.387,0.613] -> sdss_glon_100.csv
[07/14] glon_105: RA[3.717,4.717] Dec[0.371,1.371] -> sdss_glon_105.csv
[08/14] glon_110: RA[6.088,7.088] Dec[0.949,1.949] -> sdss_glon_110.csv
[09/14] glon_115: RA[8.496,9.496] Dec[1.342,2.342] -> sdss_glon_115.csv
[10/14] glon_120: RA[10.928,11.928] Dec[1.547,2.547] -> sdss_glon_120.csv
[11/14] glon_125: RA[13.369,14.369] Dec[1.564,2.564] -> sdss_glon_125.csv
[12/14] glon_130: RA[15.804,16.804] Dec[1.391,2.391] -> sdss_glon_130.csv
[13/14] glon_135: 